# 02 — Closures & Decorators
**References:** Ramalho *Fluent Python* Ch. 7–9 · Beazley *Python Cookbook* Ch. 9

## Narrative thread
```
First-class functions -> Closures -> Free variables -> Decorator basics -> Decorator factories -> functools -> Class decorators
```

## 1. First-class functions and closures

In Python, functions are **first-class objects**: they can be stored in variables, passed as arguments, returned from other functions, and stored in data structures.

A **closure** is a function that retains the binding of a **free variable** from the enclosing scope even after that scope is gone. The binding is stored in `__closure__`.

In [ ]:
import dis
import functools
import time
import weakref
from typing import Callable, TypeVar, ParamSpec

P = ParamSpec('P')
T = TypeVar('T')

# ── Closure: the make_averager example ──────────────────────────────────
def make_averager():
    series = []          # free variable — lives in the closure cell

    def averager(new_value):
        series.append(new_value)
        total = sum(series)
        return total / len(series)

    return averager

avg = make_averager()
print(f'{avg(10):.1f}')   # 10.0
print(f'{avg(11):.1f}')   # 10.5
print(f'{avg(15):.1f}')   # 12.0

# Inspect the closure
print(f'\nFree variables: {avg.__code__.co_freevars}')
print(f'Closure cells:  {avg.__closure__}')
print(f'Cell content:   {avg.__closure__[0].cell_contents}')

print()
# ── nonlocal: rebinding free variables ──────────────────────────────────
def make_averager_v2():
    count = 0
    total = 0.0

    def averager(new_value):
        nonlocal count, total    # without nonlocal, count += 1 raises UnboundLocalError
        count += 1
        total += new_value
        return total / count

    return averager

avg2 = make_averager_v2()
for v in [10, 11, 15]:
    print(f'  avg({v}) = {avg2(v):.2f}')

## 2. The decorator pattern

A **decorator** is a callable that takes a function, wraps it, and returns the wrapped version. The `@` syntax is just syntactic sugar:

```python
@decorator
def func(): ...
# equivalent to:
func = decorator(func)
```

`functools.wraps` preserves the original function's `__name__`, `__doc__`, `__annotations__`, and `__wrapped__` — always use it.

In [ ]:
# ── Production-grade decorator template ──────────────────────────────────
def timer(func: Callable[P, T]) -> Callable[P, T]:
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
        t0     = time.perf_counter()
        result = func(*args, **kwargs)
        dt     = time.perf_counter() - t0
        print(f'[timer] {func.__qualname__!r} ran in {dt*1000:.3f} ms')
        return result
    return wrapper

@timer
def slow_sum(n):
    """Sum integers 0..n-1."""
    return sum(range(n))

result = slow_sum(10_000_000)
print(f'Result: {result}')
print(f'Name preserved: {slow_sum.__name__!r}')
print(f'Doc preserved:  {slow_sum.__doc__!r}')
print(f'Wrapped:        {slow_sum.__wrapped__}')

print()
# ── Decorator factory (parametrized decorator) ───────────────────────────
def retry(max_attempts: int = 3, exceptions: tuple = (Exception,)):
    """Retry a function up to max_attempts times on given exceptions."""
    def decorator(func: Callable[P, T]) -> Callable[P, T]:
        @functools.wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
            last_exc = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    last_exc = exc
                    print(f'  [retry] attempt {attempt}/{max_attempts} failed: {exc}')
            raise last_exc
        return wrapper
    return decorator

call_count = 0

@retry(max_attempts=3, exceptions=(ValueError,))
def flaky_fn():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ValueError(f'Call {call_count} failed')
    return 'success'

print(flaky_fn())

In [ ]:
# ── Cache decorator: memoization with LRU eviction ────────────────────────
# functools.lru_cache and functools.cache (unbounded, Python 3.9+)

@functools.lru_cache(maxsize=128)
def fib(n: int) -> int:
    if n < 2: return n
    return fib(n-1) + fib(n-2)

print('Fibonacci via memoized recursion:')
print([fib(n) for n in range(15)])
print(f'Cache info: {fib.cache_info()}')
fib.cache_clear()

print()
# ── Class-based decorator (stateful) ─────────────────────────────────────
class RateLimit:
    """Allow at most `calls` calls per `period` seconds."""
    def __init__(self, calls: int = 5, period: float = 1.0):
        self.calls  = calls
        self.period = period
        self._times: list[float] = []

    def __call__(self, func: Callable[P, T]) -> Callable[P, T]:
        @functools.wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
            now = time.monotonic()
            # Remove calls outside the window
            self._times = [t for t in self._times if now - t < self.period]
            if len(self._times) >= self.calls:
                raise RuntimeError(
                    f'Rate limit exceeded: {self.calls} calls per {self.period}s'
                )
            self._times.append(now)
            return func(*args, **kwargs)
        return wrapper

@RateLimit(calls=3, period=1.0)
def api_call(endpoint: str) -> str:
    return f'Response from {endpoint}'

for i in range(3):
    print(api_call(f'/data/{i}'))

try:
    api_call('/overflow')
except RuntimeError as e:
    print(f'Caught: {e}')

print()
# ── Stacking decorators ───────────────────────────────────────────────────
@timer
@functools.lru_cache(maxsize=256)
def expensive(n: int) -> int:
    """Applied bottom-up: cache first, then timer wraps cached version."""
    return sum(i**2 for i in range(n))

expensive(10_000)
expensive(10_000)  # second call: cached, no computation

## 3. `functools` toolkit

| Tool | Purpose |
|---|---|
| `functools.wraps` | Copy metadata from wrapped function |
| `functools.lru_cache` | Memoize with LRU eviction |
| `functools.cache` | Unbounded memoize (3.9+) |
| `functools.partial` | Freeze some arguments |
| `functools.reduce` | Left fold over a sequence |
| `functools.singledispatch` | Function overloading on first argument type |
| `functools.total_ordering` | Fill comparison methods |
| `functools.cached_property` | Lazy class attribute (computed once) |

In [ ]:
from functools import singledispatch, partial, cached_property

# ── singledispatch: OOP-style overloading without inheritance ─────────────
@singledispatch
def serialize(obj) -> str:
    raise TypeError(f'Cannot serialize {type(obj).__name__}')

@serialize.register(int)
@serialize.register(float)
def _(obj) -> str:
    return f'number:{obj}'

@serialize.register(str)
def _(obj) -> str:
    return f'string:{obj!r}'

@serialize.register(list)
def _(obj) -> str:
    return f'[{', '.join(serialize(x) for x in obj)}]'

print('singledispatch:')
print(serialize(42))
print(serialize(3.14))
print(serialize('hello'))
print(serialize([1, 'two', 3.0]))

print()
# ── partial: create specialized callables ────────────────────────────────
import operator
double = partial(operator.mul, 2)
print(f'partial double: {list(map(double, range(5)))}')

# ── cached_property: lazy computed attribute ──────────────────────────────
class DataSet:
    def __init__(self, data: list[float]):
        self._data = data

    @cached_property
    def mean(self) -> float:
        print('  computing mean...')
        return sum(self._data) / len(self._data)

    @cached_property
    def variance(self) -> float:
        print('  computing variance...')
        m = self.mean
        return sum((x - m)**2 for x in self._data) / len(self._data)

ds = DataSet([1.0, 2.0, 3.0, 4.0, 5.0])
print('First access:')
print(f'  mean = {ds.mean:.2f}')
print(f'  var  = {ds.variance:.2f}')
print('Second access (no recomputation):')
print(f'  mean = {ds.mean:.2f}')

## 4. Key takeaways

| Concept | Rule |
|---|---|
| Closure | Function + free variable bindings; stored in `__closure__` |
| `nonlocal` | Required to rebind (not just mutate) a free variable |
| `@functools.wraps` | Always use — preserves metadata and `__wrapped__` |
| Decorator factory | Three levels of nesting: factory → decorator → wrapper |
| Class decorator | Better for stateful decorators; `__call__` = wrapper |
| Stacking | Applied bottom-up: `@a @b def f` → `f = a(b(f))` |
| `singledispatch` | Dispatch on first-arg type without `isinstance` chains |
| `cached_property` | Compute once per instance; works with `__dict__` |

**Next:** notebook 03 — iterators, generators, and the `itertools` algebra.